Two type of training:
1. Separate GNN Training ->  Coles with embedding layer initialized with gnn embedidngs
    * Hyperparams
        1. N_EPOCHES_BEFORE_FREEZING (freeze embedding layer initializaed with gnn embeddings) (may be any positive integer of inf)
        2. GNN_NAME (name of the gnn model)
        3. HAS_RES_CON (whether to use residual connection in gnn)
        4. WL - предполагаю, что это та же alpha, но нужно уточнить
        4. GNN_EPOCHES
        5. COLES_EPOCHES 


2. Joint Coles + GNN Training -> Coles with embedding layer initialized with gnn embedidngs
    * Hyperparams
        1. N_EPOCHES_BEFORE_FREEZING (freeze embedding layer initializaed with gnn embeddings) (may be any positive integer of inf)
        2. GNN_NAME (name of the gnn model)
        3. HAS_RES_CON (whether to use residual connection in gnn)
        4. WL - предполагаю, что это та же alpha, но нужно уточнить
        4. COLES_GNN_EPOCHES 
        5. SEPARATE_COLES_EPOCHES 
        6. `- - - -` 
        7. gamma
        8. graph task
        9. loss_wp
        10. alpha
        11. Has gnn users in coles loss

In [1]:
import sys
import os

sys.path.append(os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd()))))

In [2]:
COLES_METRICS = {'AUC test': 0.877, 'ACC test': 0.793}

In [3]:
from typing import List, Dict, Any


def assert_metric_in_data_and_number(data_lst: List[Dict[str, Any]], metric_name: str):
    for el in data_lst:
        assert metric_name in el, f"Metric name {metric_name} not found in data"
        assert isinstance(el[metric_name], (int, float)), f"Metric {metric_name} should be a number. Found {type(el[metric_name])}"


def sort_by_col(data_lst: List[Dict[str, Any]], metric_name: str):
    assert_metric_in_data_and_number(data_lst, metric_name)
    data_lst = sorted(data_lst, key=lambda x: x[metric_name], reverse=True)
    return data_lst


def bolden_top_k(data_lst: List[Dict[str, Any]], k, metric_names: List[str]):
    for metric_name in metric_names:
        assert_metric_in_data_and_number(data_lst, metric_name)

    for metric_name in metric_names:
        sorted_with_idxs = sorted(enumerate(data_lst), key=lambda x: x[1][metric_name], reverse=True)
        for i, data in sorted_with_idxs[:k]:
            data_lst[i][metric_name] = "\\textbf{" + str(data[metric_name]) + "}" 
    return data_lst


def get_idxs_where_all_metrics_superpass(data_lst, metric_name_to_limit_val: Dict[str, float]):
    idxs = []
    for i, data in enumerate(data_lst):
        superpass = True
        for metric_name, limit_val in metric_name_to_limit_val.items():
            if data[metric_name] < limit_val:
                superpass = False
                break
        if superpass:
            idxs.append(i)
    return idxs

def prefix_map_from_idx_lst(idx_lst, prefix):
    return {i: prefix for i in idx_lst}

In [4]:
# USED_EXPERIMENTS_SET = set()

In [5]:
from collections import defaultdict

from ptls_extension_2024_research.latex_table_creation import create_latex_table, get_metrics


def get_res_true_list(GNN_NAMES, WD_OPTIONS, dir_to_epoches, MAX_EPOCHES, WL, report_file_content):
    res_true_list = []

    for gnn_name in GNN_NAMES:
        for wd in WD_OPTIONS:

            model_name = f"wl-{WL}_gnn-{gnn_name}_res-True_wd-{wd}"
            for epoch in dir_to_epoches[model_name]:

                experiment_name = f'coles_gnn__pretrained_{model_name}__pretrain_epoches_{epoch}__epoches_{MAX_EPOCHES}'

                metrics = get_metrics(report_file_content, experiment_name, {'auroc': 'AUC test', 'accuracy': 'ACC test'})

                if metrics is None:
                    continue

                # USED_EXPERIMENTS_SET.add(experiment_name)

                hyper_params = {
                    "GNN": gnn_name,
                    "Has residual conncetion": "yes",
                    "wl": WL,
                    "weight decay": wd,
                    "Pretrain epochs": epoch,
                    "Epochs before unfreezing": "0",
                    "Train epochs": str(MAX_EPOCHES),
                }

                res_true_list.append({**hyper_params, **metrics})
    return res_true_list



def get_res_false_list(GNN_NAMES, WD_OPTIONS, dir_to_epoches, MAX_EPOCHES, WL, report_file_content):
    res_false_list = []

    has_res_con = False
    gnn_name = 'GraphSAGE'
    wd = 0

    model_name = f"wl-{WL}_gnn-{gnn_name}_res-{has_res_con}_wd-{wd}"
    for epoch in dir_to_epoches[model_name]:
        experiment_name = f'coles_gnn__pretrained_{model_name}__pretrain_epoches_{epoch}__epoches_{MAX_EPOCHES}'

        metrics = get_metrics(report_file_content, experiment_name, {'auroc': 'AUC test', 'accuracy': 'ACC test'})

        if metrics is None:
            continue

        # USED_EXPERIMENTS_SET.add(experiment_name)

        hyper_params = {
            "GNN": gnn_name,
            "Has residual conncetion": "yes" if has_res_con else "no",
            "wl": WL,
            "weight decay": wd,
            "Pretrain epochs": epoch,
            "Epochs before unfreezing": "0",
            "Train epochs": str(MAX_EPOCHES),
        }

        res_false_list.append({**hyper_params, **metrics})

    return res_false_list


def get_res_true_frozen_list(GNN_NAMES, WD_OPTIONS, dir_to_epoches, MAX_EPOCHES, WL, report_file_content):
    res_true_list = []

    for gnn_name in GNN_NAMES:
        for wd in WD_OPTIONS:

            model_name = f"wl-{WL}_gnn-{gnn_name}_res-True_wd-{wd}"
            for epoch in dir_to_epoches[model_name]:
                
                experiment_name = f'coles_gnn__pretrained_frozen__{model_name}__pretrain_epoches_{epoch}__epoches_{MAX_EPOCHES}'

                metrics = get_metrics(report_file_content, experiment_name, {'auroc': 'AUC test', 'accuracy': 'ACC test'})

                if metrics is None:
                    continue

                # USED_EXPERIMENTS_SET.add(experiment_name)

                hyper_params = {
                    "GNN": gnn_name,
                    "Has residual conncetion": "yes",
                    "wl": WL,
                    "weight decay": wd,
                    "Pretrain epochs": epoch,
                    "Epochs before unfreezing": r"\infty",
                    "Train epochs": str(MAX_EPOCHES),
                }

                res_true_list.append({**hyper_params, **metrics})
    return res_true_list


# def get_res_false_frozen_list(GNN_NAMES, WD_OPTIONS, dir_to_epoches, MAX_EPOCHES, WL, report_file_content):
#     res_false_list = []

#     has_res_con = False
#     gnn_name = 'GraphSAGE'
#     wd = 0

#     model_name = f"wl-{WL}_gnn-{gnn_name}_res-{has_res_con}_wd-{wd}"
#     for epoch in dir_to_epoches[model_name]:
#         experiment_name = f'coles_gnn__pretrained_frozen__{model_name}__pretrain_epoches_{epoch}__epoches_{MAX_EPOCHES}'

#         metrics = get_metrics(report_file_content, experiment_name, {'auroc': 'AUC test', 'accuracy': 'ACC test'})

#         hyper_params = {
#             "GNN": gnn_name,
#             "Has residual conncetion": "yes" if has_res_con else "no",
#             "wl": WL,
#             "Pretrain epochs": epoch,
#             "Epochs before unfreezing": r"\infty",
#             "Train epochs": str(MAX_EPOCHES),
#         }

#         res_false_list.append({**hyper_params, **metrics})

#     return res_false_list



def get_unfreeze_after_n_list(GNN_NAMES, WD_OPTIONS,
                              dir_to_epoches, MAX_EPOCHES, 
                              WL, n_epoches_before_unfreeze, 
                              report_file_content):
    res_true_list = []

    for gnn_name in GNN_NAMES:
        for wd in WD_OPTIONS:
            for has_residual_connection in [True, False]:

                model_name = f"wl-{WL}_gnn-{gnn_name}_res-{has_residual_connection}_wd-{wd}"

                if model_name not in dir_to_epoches:
                    continue

                for epoch in dir_to_epoches[model_name]:                
                    experiment_name = f'coles_gnn__pretrained_unfreeze_after_{n_epoches_before_unfreeze}_epoches__{model_name}__pretrain_epoches_{epoch}__epoches_{MAX_EPOCHES}'

                    metrics = get_metrics(report_file_content, experiment_name, {'auroc': 'AUC test', 'accuracy': 'ACC test'})

                    if metrics is None:
                        continue

                    # USED_EXPERIMENTS_SET.add(experiment_name)

                    hyper_params = {
                        "GNN": gnn_name,
                        "Has residual conncetion": "yes" if has_residual_connection else "no",
                        "wl": WL,
                        "weight decay": wd,
                        "Pretrain epochs": epoch,
                        "Epochs before unfreezing": f"{n_epoches_before_unfreeze}",
                        "Train epochs": str(MAX_EPOCHES),
                    }

                    res_true_list.append({**hyper_params, **metrics})
    return res_true_list





def get_res_true_list_from_checkpoints(GNN_NAMES, WD_OPTIONS, 
                                       dir_to_epoches, MAX_EPOCHES, 
                                       ckpt_epochs,
                                       WL, report_file_content):
    res_true_list = []

    for gnn_name in GNN_NAMES:
        for wd in WD_OPTIONS:
            for ckpt_epoch in ckpt_epochs:

                model_name = f"wl-{WL}_gnn-{gnn_name}_res-True_wd-{wd}"
                for epoch in dir_to_epoches[model_name]:

                    experiment_name = f'coles_gnn__pretrained_{model_name}__pretrain_epoches_{epoch}__epoches_{MAX_EPOCHES}__ckpt_epoch_{ckpt_epoch}'

                    print(experiment_name)

                    metrics = get_metrics(report_file_content, experiment_name, {'auroc': 'AUC test', 'accuracy': 'ACC test'})

                    if metrics is None:
                        continue

                    # USED_EXPERIMENTS_SET.add(experiment_name)

                    hyper_params = {
                        "GNN": gnn_name,
                        "Has residual conncetion": "yes",
                        "wl": WL,
                        "weight decay": wd,
                        "Pretrain epochs": epoch,
                        "Epochs before unfreezing": "0",
                        "Train epochs": str(ckpt_epoch + 1),
                    }

                    res_true_list.append({**hyper_params, **metrics})
    return res_true_list

/path/to/repo/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
dir_to_epoches = {
    "wl-0.5_gnn-GAT_res-True_wd-0": [15, 50, 100, 150, 200],
    "wl-0.5_gnn-GAT_res-True_wd-0.1": [25, 50, 75, 100, 150, 200],
    "wl-0.5_gnn-GAT_res-True_wd-0.01": [25, 50, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-False_wd-0": [50, 75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0": [75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0.1": [50, 75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0.01": [25, 50, 75, 100, 150, 200],  # был еще 200, но его не запустили
}



GNN_NAMES = ['GAT', 'GraphSAGE']
WD_OPTIONS = [0, 0.1, 0.01]

MAX_EPOCHES = 40
WL = 0.5


WEIGHTS_TYPE = 'type 2'



with open("../../results/scenario_gender_2024_research__pretrained_gnn.txt") as f:
    report_file_content = f.read()





separate_coles_and_gnn_training_experiment_dicts_list = [
    *get_res_true_list(GNN_NAMES, WD_OPTIONS, dir_to_epoches, MAX_EPOCHES, WL, report_file_content),
    *get_res_false_list(GNN_NAMES, WD_OPTIONS, dir_to_epoches, MAX_EPOCHES, WL, report_file_content),
    *get_res_true_frozen_list(GNN_NAMES, [0, 0.01], dir_to_epoches, MAX_EPOCHES, WL, report_file_content),
    *get_unfreeze_after_n_list(GNN_NAMES, WD_OPTIONS, dir_to_epoches, MAX_EPOCHES, WL, 10, report_file_content),
    *get_unfreeze_after_n_list(GNN_NAMES, WD_OPTIONS, dir_to_epoches, MAX_EPOCHES, WL, 20, report_file_content),
    *get_res_true_list_from_checkpoints(GNN_NAMES, WD_OPTIONS, dir_to_epoches, 150, 
                                        list(range(9, 150, 10)), WL, report_file_content),
]


separate_coles_and_gnn_training_experiment_dicts_list = sort_by_col(separate_coles_and_gnn_training_experiment_dicts_list, 'AUC test')
separate_coles_and_gnn_prefix_map = prefix_map_from_idx_lst(get_idxs_where_all_metrics_superpass(separate_coles_and_gnn_training_experiment_dicts_list, COLES_METRICS), "\n\\rowcolor{gray!50}\n")
separate_coles_and_gnn_training_experiment_dicts_list = bolden_top_k(separate_coles_and_gnn_training_experiment_dicts_list, 3, ['AUC test', 'ACC test'])




SEPARATE_TRAINING_EXPERIMENT_NAME = r'\makecell{Separate GNN Training\\ → \\CoLES with embedding layer\\ initialized with gnn embeddings}'

experiment_data = {
    SEPARATE_TRAINING_EXPERIMENT_NAME: {
        WEIGHTS_TYPE: separate_coles_and_gnn_training_experiment_dicts_list
    }
}


Could not find experiment coles_gnn__pretrained_wl-0.5_gnn-GraphSAGE_res-True_wd-0.01__pretrain_epoches_200__epoches_40 in file content
Could not find experiment coles_gnn__pretrained_frozen__wl-0.5_gnn-GAT_res-True_wd-0__pretrain_epoches_50__epoches_40 in file content
Could not find experiment coles_gnn__pretrained_frozen__wl-0.5_gnn-GAT_res-True_wd-0__pretrain_epoches_100__epoches_40 in file content
Could not find experiment coles_gnn__pretrained_frozen__wl-0.5_gnn-GAT_res-True_wd-0__pretrain_epoches_150__epoches_40 in file content
Could not find experiment coles_gnn__pretrained_frozen__wl-0.5_gnn-GAT_res-True_wd-0__pretrain_epoches_200__epoches_40 in file content
Could not find experiment coles_gnn__pretrained_frozen__wl-0.5_gnn-GAT_res-True_wd-0.01__pretrain_epoches_150__epoches_40 in file content
Could not find experiment coles_gnn__pretrained_frozen__wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_200__epoches_40 in file content
Could not find experiment coles_gnn__pretraine

In [7]:
# USED_EXPERIMENTS_SET

In [8]:
print(len(separate_coles_and_gnn_training_experiment_dicts_list))

98


In [9]:
# experiment_data

In [10]:
hyperparameter_header_strs = [
    r"\textbf{GNN}",  r"\textbf{Has residual conncetion}", 
    r"\textbf{$\alpha$}", r"\textbf{Weight decay}", r"\textbf{Pretrain epochs}", 
    r"\textbf{Train epochs}", r"\textbf{Epochs before unfreezing}", 
    r"\textbf{AUC test}", r"\textbf{ACC test}",  
]


hyperparameters = [
    "GNN", "Has residual conncetion", 
    "wl", "weight decay", "Pretrain epochs", 
    "Train epochs", "Epochs before unfreezing", 
    "AUC test", "ACC test"]

caption = "Comparison of different setups where GNN is pretrained."

row_prefix_dict = {
    SEPARATE_TRAINING_EXPERIMENT_NAME: {
        WEIGHTS_TYPE: separate_coles_and_gnn_prefix_map
    }
}

latex_table = create_latex_table(experiment_data, hyperparameters, hyperparameter_header_strs, caption, row_prefix_dict)

print(latex_table)

\begin{table*}[h!]
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{|c|c|c|c|c|c|c|c|c|c|c|}
\hline
\textbf{Model} & \textbf{Type of weights} & \textbf{GNN} & \textbf{Has residual conncetion} & \textbf{$\alpha$} & \textbf{Weight decay} & \textbf{Pretrain epochs} & \textbf{Train epochs} & \textbf{Epochs before unfreezing} & \textbf{AUC test} & \textbf{ACC test}\\
\hline

\rowcolor{gray!50}
\multirow{98}{*}{\makecell{Separate GNN Training\\ → \\CoLES with embedding layer\\ initialized with gnn embeddings}} &\multirow{98}{*}{type 2} & GraphSAGE & yes & 0.5 & 0 & 100 & 40 & 0 & \textbf{0.886} & 0.801 \\ \cline{3-11} 

\rowcolor{gray!50}
& &GraphSAGE & yes & 0.5 & 0 & 100 & 30 & 0 & \textbf{0.883} & \textbf{0.804} \\ \cline{3-11} 

\rowcolor{gray!50}
& &GAT & yes & 0.5 & 0 & 100 & 40 & 0 & \textbf{0.882} & 0.795 \\ \cline{3-11} 

\rowcolor{gray!50}
& &GraphSAGE & no & 0.5 & 0 & 75 & 40 & 0 & 0.882 & 0.803 \\ \cline{3-11} 

\rowcolor{gray!50}
& &GraphSAGE & yes & 0.5 & 0.01 & 50 & 40 &

In [9]:
# ALL_EXPERIMENTS = {el for el in report_file_content.split(' ') if el.startswith('coles')}
# ALL_EXPERIMENTS == USED_EXPERIMENTS_SET

In [98]:
coles_gnn_pretrained_hyperparams_list = []


with open("../../results/scenario_gender_2024_research__pretrained_coles_gnn.txt") as f:
    report_file_content = f.read()


# Define the options
INCLUDE_GNN_USERS_IN_COLES_LOSS_OPTIONS = [False]
HAS_ORIG_EMBS_OPTIONS = [False]
GNN_NAME_OPTIONS = ["GraphSAGE"]
COLES_LOSS_GAMMA_OPTIONS = [0.5]
GNN_LOSS_ALPHA_OPTIONS = [0.5]
LP_CRITERION_OPTIONS = ["MSELoss"]

MAX_EPOCHES_ORIG = 40
CKPT_EPOCHS = [4, 9, 14, 19, 29, 34]

# Nested loops to iterate over all combinations of the options
for INCLUDE_GNN_USERS_IN_COLES_LOSS in INCLUDE_GNN_USERS_IN_COLES_LOSS_OPTIONS:
    for HAS_ORIG_EMBS in HAS_ORIG_EMBS_OPTIONS:
        for GNN_NAME in GNN_NAME_OPTIONS:
            for COLES_LOSS_GAMMA in COLES_LOSS_GAMMA_OPTIONS:
                for GNN_LOSS_ALPHA in GNN_LOSS_ALPHA_OPTIONS:
                    for LP_CRITERION in LP_CRITERION_OPTIONS:
                        for CKPT_EPOCH in CKPT_EPOCHS:

                            MAX_EPOCHES = MAX_EPOCHES_ORIG - CKPT_EPOCH - 1

                            # Determine INCLUDE_GNN_USERS_IN_COLES_LOSS_STR based on the value of INCLUDE_GNN_USERS_IN_COLES_LOSS
                            if INCLUDE_GNN_USERS_IN_COLES_LOSS:
                                INCLUDE_GNN_USERS_IN_COLES_LOSS_STR = "gnn_users_in_coles_loss"
                            else:
                                INCLUDE_GNN_USERS_IN_COLES_LOSS_STR = ""

                            # Determine HAS_ORIG_EMBS_STR and TRX_SIMPLE_EMBEDDING_LAYERS based on the value of HAS_ORIG_EMBS
                            if HAS_ORIG_EMBS:
                                HAS_ORIG_EMBS_STR = "has_orig"
                                TRX_SIMPLE_EMBEDDING_LAYERS = "all_features"
                            else:
                                HAS_ORIG_EMBS_STR = "no_orig"
                                TRX_SIMPLE_EMBEDDING_LAYERS = "all_except_item_id"

                            ORIGINAL_MODEL_NAME = f"coles_gnn_weighted__w_pred___{GNN_NAME}__{HAS_ORIG_EMBS_STR}__alpha_{GNN_LOSS_ALPHA}__gamma_{COLES_LOSS_GAMMA}__{INCLUDE_GNN_USERS_IN_COLES_LOSS_STR}__lp_criterion_{LP_CRITERION}__{MAX_EPOCHES_ORIG}_epoches"

                            experiment_name = f"check_coles_only__pretrained_epoches_{CKPT_EPOCH}__coles_gnn_weighted__w_pred__no_gnn_{GNN_NAME}__{HAS_ORIG_EMBS_STR}__alpha_{GNN_LOSS_ALPHA}__gamma_{COLES_LOSS_GAMMA}__{INCLUDE_GNN_USERS_IN_COLES_LOSS_STR}__lp_criterion_{LP_CRITERION}__{MAX_EPOCHES_ORIG}_epoches__epoches_after_ckpt_{MAX_EPOCHES}__try_1"


                            metrics = get_metrics(report_file_content, experiment_name, {'auroc': 'AUC test', 'accuracy': 'ACC test'})

                            if metrics is None:
                                continue

                            # USED_EXPERIMENTS_SET.add(experiment_name)

                            hyper_params = {
                                "GNN": GNN_NAME,
                                "Joint pretrain epochs": CKPT_EPOCH + 1,
                                "Epochs before unfreezing": "0",
                                "Train epochs": MAX_EPOCHES,
                                "gamma": COLES_LOSS_GAMMA,
                                "graph_task": "LP+WR",
                                "loss_wp": LP_CRITERION,
                                "alpha": GNN_LOSS_ALPHA,
                                "Has gnn users in coles loss": "yes" if INCLUDE_GNN_USERS_IN_COLES_LOSS else "no"
                            }

                            coles_gnn_pretrained_hyperparams_list.append({**hyper_params, **metrics})
                         

WEIGHTS_TYPE = "type 2"


coles_gnn_pretrained_hyperparams_list = sort_by_col(coles_gnn_pretrained_hyperparams_list, 'AUC test')
coles_gnn_pretrained_prefix_map = prefix_map_from_idx_lst(get_idxs_where_all_metrics_superpass(coles_gnn_pretrained_hyperparams_list, COLES_METRICS), "\n\\rowcolor{gray!50}\n")
coles_gnn_pretrained_hyperparams_list = bolden_top_k(coles_gnn_pretrained_hyperparams_list, 3, ['AUC test', 'ACC test'])

experiment_name = r'\makecell{Joint CoLES & GNN Training\\ → \\CoLES with embedding layer\\ initialized with gnn embeddings}'

coles_gnn_pretrained_row_prefix_dict = {
    experiment_name: {
        WEIGHTS_TYPE: coles_gnn_pretrained_prefix_map
    }
}


coles_gnn_pretrained_experiment_data = {
    experiment_name: {
        WEIGHTS_TYPE: coles_gnn_pretrained_hyperparams_list
    }
}

hyperparameter_header_strs = [
    r"\textbf{GNN}",  r"\textbf{Joint pretrain epochs}",
    r"\textbf{Epochs before unfreezing}", r"\textbf{Train epochs}",
    r"\textbf{$\gamma$}", r"\textbf{graph task}", r"\textbf{loss wp}", r"\textbf{$\alpha$}",
    r"\textbf{Has gnn users in coles loss}", r"\textbf{AUC test}", r"\textbf{ACC test}",
]

hyperparameters = [
    "GNN", "Joint pretrain epochs", "Epochs before unfreezing", "Train epochs",
    "gamma", "graph_task", "loss_wp", "alpha",
    "Has gnn users in coles loss", "AUC test", "ACC test"
]

caption = "Joint CoLES & GNN pretraining then CoLES only training."

coles_gnn_pretrained_latex_table = create_latex_table(coles_gnn_pretrained_experiment_data, hyperparameters, hyperparameter_header_strs, caption, coles_gnn_pretrained_row_prefix_dict)

print(coles_gnn_pretrained_latex_table)

\begin{table*}[h!]
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{|c|c|c|c|c|c|c|c|c|c|c|c|c|}
\hline
\textbf{Model} & \textbf{Type of weights} & \textbf{GNN} & \textbf{Joint pretrain epochs} & \textbf{Epochs before unfreezing} & \textbf{Train epochs} & \textbf{$\gamma$} & \textbf{graph task} & \textbf{loss wp} & \textbf{$\alpha$} & \textbf{Has gnn users in coles loss} & \textbf{AUC test} & \textbf{ACC test}\\
\hline

\rowcolor{gray!50}
\multirow{6}{*}{\makecell{Joint CoLES & GNN Training\\ → \\CoLES with embedding layer\\ initialized with gnn embeddings}} &\multirow{6}{*}{type 2} & GraphSAGE & 15 & 0 & 25 & 0.5 & LP+WR & MSELoss & 0.5 & no & \textbf{0.879} & \textbf{0.796} \\ \cline{3-13} 
& &GraphSAGE & 20 & 0 & 20 & 0.5 & LP+WR & MSELoss & 0.5 & no & \textbf{0.877} & 0.789 \\ \cline{3-13} 
& &GraphSAGE & 30 & 0 & 10 & 0.5 & LP+WR & MSELoss & 0.5 & no & \textbf{0.873} & \textbf{0.79} \\ \cline{3-13} 
& &GraphSAGE & 35 & 0 & 5 & 0.5 & LP+WR & MSELoss & 0.5 & no & 0.872 & \text